# Proyecto final — Predicción de conversión y Optimización Bayesiana
### Caso académico ficticio inspirado en Movistar

**Curso:** Machine Learning & Deep Learning — ESAN  
**Objetivo:** construir un pipeline reproducible que:
1. genere un dataset sintético de campañas digitales,
2. prediga la probabilidad de conversión,
3. compare Logistic Regression, Random Forest y XGBoost,
4. seleccione un modelo ganador,
5. use Optimización Bayesiana para recomendar la siguiente configuración de campaña,
6. compare Optimización Bayesiana vs. Random Search.

> **Importante:** todos los datos de este notebook son sintéticos. No se utilizan datos reales ni sensibles de Movistar.

## 0. Cómo usar este notebook

Ejecutar las celdas en orden desde **Entorno de ejecución → Ejecutar todas**.

La lógica del proyecto es:

**Datos sintéticos → EDA → Split temporal → Preparación → Modelos ML → Evaluación → Score de negocio → Proceso Gaussiano → UCB → BO vs Random Search → Recomendación final**

Las cifras finales del PPT deberían actualizarse con los resultados que efectivamente produzca este notebook.

In [ ]:
# En Google Colab, XGBoost suele venir instalado.
# Esta celda asegura que esté disponible.
!pip -q install xgboost

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score,
    recall_score, f1_score, brier_score_loss,
    confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from sklearn.preprocessing import OneHotEncoder as OHE

from xgboost import XGBClassifier

SEED = 42
rng = np.random.default_rng(SEED)

pd.set_option("display.max_columns", 100)

# 1. Generación del dataset sintético

La **unidad de análisis** será una interacción cliente–campaña–canal digital.

La variable objetivo será:

- `conversion_exitosa = 1`: el cliente completó la acción objetivo.
- `conversion_exitosa = 0`: no la completó.

La simulación incorpora relaciones intencionales para que el dataset tenga lógica de negocio: por ejemplo, ciertos horarios, beneficios, segmentos y flujos afectan la probabilidad de conversión.

In [ ]:
N = 50_000

segmentos = ["Jovenes", "Adultos", "Familias", "Premium"]
canales = ["WhatsApp", "Chatbot Web", "SMS", "Email"]
wordings = ["Directo", "Digital", "Beneficio", "Urgencia"]
beneficios = ["Sin beneficio", "Descuento 10%", "Descuento 20%", "Bono datos"]
flujos = ["Corto", "Medio", "Largo"]
dias = ["Lunes", "Martes", "Miercoles", "Jueves", "Viernes", "Sabado", "Domingo"]

df = pd.DataFrame({
    "mes": rng.integers(1, 13, N),
    "segmento": rng.choice(segmentos, N, p=[0.34, 0.31, 0.23, 0.12]),
    "antiguedad_meses": np.clip(rng.normal(34, 20, N).round(), 1, 120).astype(int),
    "plan_soles": np.clip(rng.normal(79, 35, N), 29, 250).round(2),
    "interacciones_previas": np.clip(rng.poisson(3.2, N), 0, 18),
    "campanas_ultimos_30d": np.clip(rng.poisson(2.1, N), 0, 10),
    "hora": rng.integers(8, 22, N),
    "dia_semana": rng.choice(dias, N),
    "canal": rng.choice(canales, N, p=[0.44, 0.25, 0.16, 0.15]),
    "wording": rng.choice(wordings, N),
    "beneficio": rng.choice(beneficios, N, p=[0.30, 0.28, 0.22, 0.20]),
    "flujo": rng.choice(flujos, N, p=[0.42, 0.36, 0.22]),
    "pasos_flujo": 0,
    "tiempo_respuesta_seg": np.clip(rng.lognormal(mean=3.4, sigma=0.5, size=N), 5, 240).round(1),
    "dispositivo": rng.choice(["Android", "iOS", "Desktop"], N, p=[0.58, 0.28, 0.14]),
    "region": rng.choice(["Lima", "Norte", "Centro", "Sur", "Oriente"], N, p=[0.48, 0.18, 0.12, 0.14, 0.08]),
    "cliente_digital": rng.choice([0,1], N, p=[0.28,0.72]),
    "visitas_app_30d": np.clip(rng.poisson(5.0, N), 0, 30),
    "reclamos_90d": np.clip(rng.poisson(0.55, N), 0, 5),
    "saldo_promedio": np.clip(rng.normal(24, 18, N), 0, 120).round(2)
})

# Pasos de flujo coherentes con el tipo de flujo
df["pasos_flujo"] = np.select(
    [df["flujo"].eq("Corto"), df["flujo"].eq("Medio"), df["flujo"].eq("Largo")],
    [rng.integers(2,4,N), rng.integers(4,6,N), rng.integers(6,9,N)]
)

# Función latente de conversión
z = np.full(N, -2.15)

# Segmento
z += df["segmento"].map({
    "Jovenes": 0.22, "Adultos": 0.06, "Familias": 0.02, "Premium": 0.15
}).values

# Horario: mejor alrededor de 18-20h
z += np.where(df["hora"].between(18,20), 0.42, 0)
z += np.where(df["hora"].between(12,14), 0.12, 0)
z += np.where(df["hora"] <= 9, -0.18, 0)

# Canal
z += df["canal"].map({
    "WhatsApp": 0.18, "Chatbot Web": 0.25, "SMS": -0.08, "Email": -0.05
}).values

# Wording
z += df["wording"].map({
    "Directo": 0.04, "Digital": 0.18, "Beneficio": 0.14, "Urgencia": -0.03
}).values

# Beneficio
z += df["beneficio"].map({
    "Sin beneficio": -0.18,
    "Descuento 10%": 0.14,
    "Descuento 20%": 0.35,
    "Bono datos": 0.18
}).values

# Flujo
z += df["flujo"].map({"Corto": 0.28, "Medio": 0.05, "Largo": -0.28}).values

# Características cliente
z += 0.10 * df["cliente_digital"].values
z += 0.010 * np.minimum(df["visitas_app_30d"].values, 12)
z += 0.0025 * np.minimum(df["antiguedad_meses"].values, 60)
z += 0.0015 * (df["plan_soles"].values - 70)

# Fatiga y fricción
z -= 0.12 * df["campanas_ultimos_30d"].values
z -= 0.10 * df["reclamos_90d"].values
z -= 0.015 * np.maximum(df["tiempo_respuesta_seg"].values - 30, 0) / 10

# Interacciones útiles
z += np.where(
    (df["segmento"].eq("Jovenes")) &
    (df["hora"].between(18,20)) &
    (df["beneficio"].eq("Descuento 20%")) &
    (df["flujo"].eq("Corto")),
    0.40, 0
)

# Pequeño drift temporal
z += 0.018 * (df["mes"].values - 6)

prob_conv = 1 / (1 + np.exp(-z))
df["conversion_exitosa"] = rng.binomial(1, prob_conv)

# KPIs posteriores, SOLO para EDA descriptivo, no para modelar conversión
df["ctr"] = np.clip(0.10 + 0.55*prob_conv + rng.normal(0,0.08,N), 0, 1)
df["contencion"] = np.clip(0.18 + 0.40*(df["flujo"].eq("Corto")).astype(float) + rng.normal(0,0.12,N), 0, 1)
df["csat"] = np.clip(
    3.2 + 0.7*df["conversion_exitosa"] - 0.18*df["reclamos_90d"] + rng.normal(0,0.65,N),
    1, 5
).round(1)

df.head()

In [ ]:
print("Filas:", len(df))
print("Columnas:", len(df.columns))
print("Conversión media:", round(df["conversion_exitosa"].mean(), 4))
print("\nDistribución del target:")
display(df["conversion_exitosa"].value_counts(normalize=True).rename("proporcion"))

## Nota metodológica sobre leakage

`ctr`, `contencion` y `csat` se generan como resultados del journey y se usarán únicamente con fines descriptivos.

**No se incluirán como variables predictoras de `conversion_exitosa`**, porque podrían contener información posterior o simultánea al resultado y producir fuga de información.

# 2. Análisis Exploratorio de Datos (EDA)

In [ ]:
conv_hora = df.groupby("hora")["conversion_exitosa"].mean()

plt.figure(figsize=(9,4))
plt.plot(conv_hora.index, conv_hora.values, marker="o")
plt.title("Conversión media por hora")
plt.xlabel("Hora")
plt.ylabel("Tasa de conversión")
plt.grid(alpha=0.25)
plt.show()

In [ ]:
conv_segmento = df.groupby("segmento")["conversion_exitosa"].mean().sort_values(ascending=False)

plt.figure(figsize=(8,4))
conv_segmento.plot(kind="bar")
plt.title("Conversión media por segmento")
plt.ylabel("Tasa de conversión")
plt.xlabel("")
plt.xticks(rotation=0)
plt.show()

In [ ]:
conv_beneficio = df.groupby("beneficio")["conversion_exitosa"].mean().sort_values(ascending=False)

plt.figure(figsize=(9,4))
conv_beneficio.plot(kind="bar")
plt.title("Conversión media por beneficio")
plt.ylabel("Tasa de conversión")
plt.xlabel("")
plt.xticks(rotation=20)
plt.show()

In [ ]:
conv_flujo = df.groupby("flujo")["conversion_exitosa"].mean().sort_values(ascending=False)

plt.figure(figsize=(7,4))
conv_flujo.plot(kind="bar")
plt.title("Conversión media por tipo de flujo")
plt.ylabel("Tasa de conversión")
plt.xlabel("")
plt.xticks(rotation=0)
plt.show()

In [ ]:
kpis = pd.Series({
    "CTR medio": df["ctr"].mean(),
    "Conversión": df["conversion_exitosa"].mean(),
    "Contención media": df["contencion"].mean(),
    "CSAT >= 4": (df["csat"] >= 4).mean()
})

display((kpis*100).round(1).rename("%"))

# 3. Split temporal

Para simular un escenario de predicción futura:

- **Train:** meses 1–8
- **Validation:** meses 9–10
- **Test:** meses 11–12

No se mezclan aleatoriamente los periodos, porque queremos que el modelo aprenda del pasado y sea evaluado en periodos posteriores.

In [ ]:
train = df[df["mes"].between(1,8)].copy()
valid = df[df["mes"].between(9,10)].copy()
test  = df[df["mes"].between(11,12)].copy()

target = "conversion_exitosa"

features = [
    "segmento", "antiguedad_meses", "plan_soles", "interacciones_previas",
    "campanas_ultimos_30d", "hora", "dia_semana", "canal", "wording",
    "beneficio", "flujo", "pasos_flujo", "tiempo_respuesta_seg",
    "dispositivo", "region", "cliente_digital", "visitas_app_30d",
    "reclamos_90d", "saldo_promedio"
]

X_train, y_train = train[features], train[target]
X_valid, y_valid = valid[features], valid[target]
X_test,  y_test  = test[features],  test[target]

print(len(train), len(valid), len(test))

# 4. Preprocesamiento

- Variables categóricas → `OneHotEncoder`
- Variables numéricas → imputación + escalamiento

El mismo preprocesamiento se incorpora dentro de un `Pipeline` para evitar inconsistencias entre entrenamiento y evaluación.

In [ ]:
categorical_cols = X_train.select_dtypes(include="object").columns.tolist()
numeric_cols = [c for c in features if c not in categorical_cols]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_cols),
    ("cat", categorical_pipe, categorical_cols)
])

print("Categóricas:", categorical_cols)
print("Numéricas:", numeric_cols)

# 5. Modelos de Machine Learning

Compararemos:

1. **Logistic Regression** → baseline interpretable.
2. **Random Forest** → modelo no lineal basado en múltiples árboles.
3. **XGBoost** → boosting de árboles, capaz de capturar relaciones e interacciones complejas.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1200, class_weight=None, random_state=SEED
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=250, max_depth=10, min_samples_leaf=12,
        n_jobs=-1, random_state=SEED
    ),
    "XGBoost": XGBClassifier(
        n_estimators=320,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        eval_metric="logloss",
        random_state=SEED,
        n_jobs=-1
    )
}

## 5.1 Función de evaluación

El threshold **no se fija automáticamente en 0.50**.

Primero se busca en **Validation** el threshold que maximiza F1. Luego ese mismo threshold se aplica una sola vez al conjunto **Test**.

In [ ]:
def best_f1_threshold(y_true, probas):
    thresholds = np.linspace(0.05, 0.80, 151)
    rows = []
    for t in thresholds:
        pred = (probas >= t).astype(int)
        rows.append((t, f1_score(y_true, pred)))
    best_t, best_f1 = max(rows, key=lambda x: x[1])
    return best_t, best_f1

def evaluate_probs(y_true, probas, threshold):
    pred = (probas >= threshold).astype(int)
    return {
        "AUC": roc_auc_score(y_true, probas),
        "PR-AUC": average_precision_score(y_true, probas),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "Brier": brier_score_loss(y_true, probas),
        "Threshold": threshold
    }

In [ ]:
fitted = {}
results = []

for name, model in models.items():
    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    valid_proba = pipe.predict_proba(X_valid)[:,1]
    threshold, _ = best_f1_threshold(y_valid, valid_proba)

    test_proba = pipe.predict_proba(X_test)[:,1]
    metrics = evaluate_probs(y_test, test_proba, threshold)

    fitted[name] = pipe
    results.append({"Modelo": name, **metrics})

results_df = pd.DataFrame(results).set_index("Modelo")
display(results_df.round(4))

# 6. Selección del modelo ganador

La elección no debe basarse únicamente en AUC.

Para este proyecto interesa:
- buena discriminación (`AUC`),
- desempeño sobre la clase positiva (`PR-AUC`),
- equilibrio Precision/Recall (`F1`),
- calidad de probabilidades (`Brier`, menor es mejor).

La celda siguiente selecciona como ganador el modelo con mayor **PR-AUC**, usando AUC y Brier como criterios complementarios.

In [ ]:
ranking = results_df.copy()
ranking["score_seleccion"] = (
    0.45 * ranking["PR-AUC"].rank(pct=True) +
    0.30 * ranking["AUC"].rank(pct=True) +
    0.15 * ranking["F1"].rank(pct=True) +
    0.10 * (-ranking["Brier"]).rank(pct=True)
)

winner_name = ranking["score_seleccion"].idxmax()
winner = fitted[winner_name]
winner_threshold = results_df.loc[winner_name, "Threshold"]

print("Modelo ganador:", winner_name)
display(ranking.sort_values("score_seleccion", ascending=False).round(4))

# 7. Evaluación consistente del modelo ganador

**Importante:** matriz de confusión y métricas se calculan aquí usando exactamente:
- el mismo conjunto Test,
- las mismas probabilidades,
- el mismo threshold seleccionado en Validation.

Así evitamos inconsistencias entre diapositivas.

In [ ]:
winner_test_proba = winner.predict_proba(X_test)[:,1]
winner_test_pred = (winner_test_proba >= winner_threshold).astype(int)

winner_metrics = evaluate_probs(y_test, winner_test_proba, winner_threshold)

print("Modelo:", winner_name)
for k, v in winner_metrics.items():
    print(f"{k}: {v:.4f}")

In [ ]:
cm = confusion_matrix(y_test, winner_test_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0,1])
disp.plot(values_format="d")
plt.title(f"Matriz de confusión — {winner_name}")
plt.show()

tn, fp, fn, tp = cm.ravel()
print({"TN": tn, "FP": fp, "FN": fn, "TP": tp})

In [ ]:
RocCurveDisplay.from_predictions(y_test, winner_test_proba)
plt.title(f"ROC — {winner_name}")
plt.show()

PrecisionRecallDisplay.from_predictions(y_test, winner_test_proba)
plt.title(f"Precision-Recall — {winner_name}")
plt.show()

# 8. Importancia de variables

Para XGBoost y Random Forest podemos recuperar importancias después del `OneHotEncoder`.

Si el ganador fuese Logistic Regression, interpretaremos la magnitud de los coeficientes.

In [ ]:
prep_fitted = winner.named_steps["prep"]
model_fitted = winner.named_steps["model"]

feature_names = prep_fitted.get_feature_names_out()

if hasattr(model_fitted, "feature_importances_"):
    imp = pd.DataFrame({
        "variable": feature_names,
        "importancia": model_fitted.feature_importances_
    }).sort_values("importancia", ascending=False).head(20)
else:
    imp = pd.DataFrame({
        "variable": feature_names,
        "importancia": np.abs(model_fitted.coef_[0])
    }).sort_values("importancia", ascending=False).head(20)

display(imp)

plt.figure(figsize=(9,6))
plt.barh(imp["variable"][::-1], imp["importancia"][::-1])
plt.title(f"Top variables — {winner_name}")
plt.xlabel("Importancia")
plt.show()

# 9. Capa prescriptiva: espacio de campañas

Machine Learning responde:

> **¿qué probabilidad de conversión estimamos?**

Optimización Bayesiana responde:

> **¿qué configuración conviene evaluar a continuación?**

Construiremos un espacio discreto de configuraciones manipulables:
- segmento,
- hora,
- canal,
- wording,
- beneficio,
- flujo.

Para evaluar cada configuración, estimaremos la conversión sobre un **perfil representativo de cliente** y construiremos un score ajustado por costo y presión de contacto.

In [ ]:
from itertools import product

space_segmento = ["Jovenes", "Adultos", "Familias", "Premium"]
space_hora = [10, 13, 16, 19, 21]
space_canal = ["WhatsApp", "Chatbot Web", "SMS", "Email"]
space_wording = ["Directo", "Digital", "Beneficio", "Urgencia"]
space_beneficio = ["Sin beneficio", "Descuento 10%", "Descuento 20%", "Bono datos"]
space_flujo = ["Corto", "Medio", "Largo"]

campaign_space = pd.DataFrame(list(product(
    space_segmento, space_hora, space_canal,
    space_wording, space_beneficio, space_flujo
)), columns=["segmento","hora","canal","wording","beneficio","flujo"])

print("Número de configuraciones candidatas:", len(campaign_space))
campaign_space.head()

## 9.1 Función de score de negocio

Para la simulación:

**score = probabilidad estimada de conversión − penalización por costo − penalización por contacto**

Las penalizaciones son supuestos académicos transparentes, no costos reales de Movistar.

In [ ]:
# Perfil representativo para puntuar configuraciones
profile = {
    "antiguedad_meses": int(df["antiguedad_meses"].median()),
    "plan_soles": float(df["plan_soles"].median()),
    "interacciones_previas": int(df["interacciones_previas"].median()),
    "campanas_ultimos_30d": 2,
    "dia_semana": "Jueves",
    "pasos_flujo": 3,
    "tiempo_respuesta_seg": float(df["tiempo_respuesta_seg"].median()),
    "dispositivo": "Android",
    "region": "Lima",
    "cliente_digital": 1,
    "visitas_app_30d": int(df["visitas_app_30d"].median()),
    "reclamos_90d": 0,
    "saldo_promedio": float(df["saldo_promedio"].median())
}

cost_benefit = {
    "Sin beneficio": 0.000,
    "Descuento 10%": 0.020,
    "Descuento 20%": 0.055,
    "Bono datos": 0.030
}
cost_channel = {
    "WhatsApp": 0.010,
    "Chatbot Web": 0.006,
    "SMS": 0.015,
    "Email": 0.004
}
cost_flow = {
    "Corto": 0.000,
    "Medio": 0.006,
    "Largo": 0.015
}

def build_scoring_rows(config_df):
    rows = []
    for _, r in config_df.iterrows():
        row = dict(profile)
        row.update(r.to_dict())
        row["pasos_flujo"] = {"Corto":3, "Medio":5, "Largo":7}[row["flujo"]]
        rows.append(row)
    return pd.DataFrame(rows)[features]

def business_score(config_df):
    scoring_X = build_scoring_rows(config_df)
    p = winner.predict_proba(scoring_X)[:,1]
    penalty = (
        config_df["beneficio"].map(cost_benefit).values +
        config_df["canal"].map(cost_channel).values +
        config_df["flujo"].map(cost_flow).values
    )
    return p, p - penalty

campaign_space["prob_conversion_estimada"], campaign_space["score_negocio"] = business_score(campaign_space)

display(campaign_space.sort_values("score_negocio", ascending=False).head(10))

# 10. Optimización Bayesiana con Proceso Gaussiano + UCB

El **Proceso Gaussiano (GP)** funcionará como modelo sustituto.

A partir de las configuraciones ya evaluadas, estima para cada alternativa:
- una media esperada `μ(x)`,
- una incertidumbre `σ(x)`.

Usaremos **UCB (Upper Confidence Bound)**:

`UCB = μ(x) + κ · σ(x)`

- `μ(x)` favorece explotación,
- `σ(x)` favorece exploración,
- `κ` controla cuánto valor damos a explorar.

In [ ]:
bo_cols = ["segmento","hora","canal","wording","beneficio","flujo"]

bo_encoder = OHE(handle_unknown="ignore", sparse_output=False)
X_space_encoded = bo_encoder.fit_transform(campaign_space[bo_cols])

y_objective = campaign_space["score_negocio"].values

kernel = (
    ConstantKernel(1.0, constant_value_bounds="fixed") *
    Matern(length_scale=1.0, nu=2.5) +
    WhiteKernel(noise_level=1e-5, noise_level_bounds="fixed")
)

def run_bayesian_optimization(
    X_encoded, y_true, n_trials=30, n_initial=5, kappa=1.5, seed=42
):
    rng_local = np.random.default_rng(seed)
    n = len(y_true)

    observed = list(rng_local.choice(n, size=n_initial, replace=False))
    history = []

    for idx in observed:
        history.append({
            "iteracion": len(history)+1,
            "indice": idx,
            "score": y_true[idx],
            "best_score": max(y_true[i] for i in observed[:len(history)])
        })

    while len(observed) < n_trials:
        gp = GaussianProcessRegressor(
            kernel=kernel,
            normalize_y=True,
            random_state=seed,
            n_restarts_optimizer=0
        )
        gp.fit(X_encoded[observed], y_true[observed])

        mu, std = gp.predict(X_encoded, return_std=True)
        ucb = mu + kappa * std

        ucb[observed] = -np.inf
        next_idx = int(np.argmax(ucb))
        observed.append(next_idx)

        history.append({
            "iteracion": len(observed),
            "indice": next_idx,
            "score": y_true[next_idx],
            "best_score": max(y_true[i] for i in observed)
        })

    return pd.DataFrame(history), observed

bo_history, bo_indices = run_bayesian_optimization(
    X_space_encoded, y_objective, n_trials=30, n_initial=5, kappa=1.5, seed=SEED
)

bo_history.head(10)

# 11. Benchmark: Optimización Bayesiana vs Random Search

Ambos métodos reciben el mismo presupuesto de **30 pruebas**.

Para evitar que una sola semilla aleatoria distorsione la conclusión, repetiremos la comparación varias veces.

In [ ]:
def run_random_search(y_true, n_trials=30, seed=42):
    rng_local = np.random.default_rng(seed)
    idx = rng_local.choice(len(y_true), size=n_trials, replace=False)
    vals = y_true[idx]
    return np.maximum.accumulate(vals)

def get_bo_curve(seed):
    hist, _ = run_bayesian_optimization(
        X_space_encoded, y_objective,
        n_trials=30, n_initial=5, kappa=1.5, seed=seed
    )
    return hist["best_score"].values

reps = 20
bo_curves = np.vstack([get_bo_curve(100+s) for s in range(reps)])
rs_curves = np.vstack([run_random_search(y_objective, 30, 100+s) for s in range(reps)])

bo_mean = bo_curves.mean(axis=0)
rs_mean = rs_curves.mean(axis=0)

plt.figure(figsize=(9,5))
plt.plot(range(1,31), bo_mean, marker="o", markersize=3, label="Optimización Bayesiana")
plt.plot(range(1,31), rs_mean, marker="o", markersize=3, label="Random Search")
plt.xlabel("Número de pruebas")
plt.ylabel("Mejor score encontrado")
plt.title("BO vs Random Search — promedio de 20 repeticiones")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

In [ ]:
global_best = y_objective.max()
target_95 = 0.95 * global_best

def first_reach(curve, target):
    hits = np.where(curve >= target)[0]
    return int(hits[0] + 1) if len(hits) else np.nan

bo_reach = [first_reach(c, target_95) for c in bo_curves]
rs_reach = [first_reach(c, target_95) for c in rs_curves]

benchmark = pd.DataFrame({
    "Método": ["Optimización Bayesiana", "Random Search"],
    "Mejor score medio al final": [bo_curves[:,-1].mean(), rs_curves[:,-1].mean()],
    "Iteración media para alcanzar 95% del óptimo": [
        np.nanmean(bo_reach), np.nanmean(rs_reach)
    ]
})

display(benchmark.round(4))

# 12. Recomendación final de campaña

Seleccionamos la mejor configuración encontrada por la ejecución principal de BO.

**Importante:** esta recomendación es una prioridad de prueba, no una decisión automática de producción.

In [ ]:
best_bo_idx = bo_history.loc[bo_history["score"].idxmax(), "indice"]
best_config = campaign_space.loc[int(best_bo_idx)]

print("CONFIGURACIÓN RECOMENDADA POR BO")
print("-" * 45)
for c in bo_cols:
    print(f"{c}: {best_config[c]}")
print(f"Probabilidad estimada de conversión: {best_config['prob_conversion_estimada']:.1%}")
print(f"Score de negocio ajustado: {best_config['score_negocio']:.4f}")

# 13. Simulador ejecutivo

Esta función permite ingresar una configuración y obtener:
- probabilidad estimada de conversión,
- score de negocio ajustado.

Puede utilizarse en la sustentación para demostrar cómo el modelo convierte variables de negocio en una recomendación cuantitativa.

In [ ]:
def evaluar_campana(
    segmento="Jovenes",
    hora=19,
    canal="Chatbot Web",
    wording="Digital",
    beneficio="Descuento 20%",
    flujo="Corto"
):
    config = pd.DataFrame([{
        "segmento": segmento,
        "hora": hora,
        "canal": canal,
        "wording": wording,
        "beneficio": beneficio,
        "flujo": flujo
    }])

    p, score = business_score(config)

    return pd.DataFrame({
        "segmento": [segmento],
        "hora": [hora],
        "canal": [canal],
        "wording": [wording],
        "beneficio": [beneficio],
        "flujo": [flujo],
        "prob_conversion_estimada": [p[0]],
        "score_negocio": [score[0]]
    })

evaluar_campana()

# 14. Exportación de resultados

Estas celdas generan archivos que luego pueden subirse a GitHub:

- `dataset_sintetico.csv`
- `metricas_modelos.csv`
- `benchmark_bo_vs_random.csv`
- `recomendacion_bo.csv`

In [ ]:
df.to_csv("dataset_sintetico.csv", index=False)
results_df.to_csv("metricas_modelos.csv")
benchmark.to_csv("benchmark_bo_vs_random.csv", index=False)
campaign_space.loc[[int(best_bo_idx)]].to_csv("recomendacion_bo.csv", index=False)

print("Archivos generados correctamente.")

# 15. Conclusiones metodológicas

1. El problema predictivo es una **clasificación binaria**.
2. Se utiliza un **split temporal** para simular predicción futura.
3. Se comparan tres modelos con métricas consistentes.
4. El threshold se define en Validation y se evalúa una sola vez en Test.
5. La capa de Machine Learning estima probabilidad de conversión.
6. La capa de Optimización Bayesiana utiliza un **Proceso Gaussiano** como modelo sustituto.
7. **UCB** balancea exploración y explotación.
8. BO y Random Search reciben el mismo presupuesto de pruebas.
9. La recomendación final debe validarse mediante un A/B test real antes de producción.
10. Los resultados son académicos porque se basan en datos sintéticos.

# 16. Próximos pasos para la entrega

Después de ejecutar este notebook:

1. Revisar las métricas reales obtenidas.
2. Actualizar el PPT para que coincida exactamente con las salidas.
3. Guardar el notebook como `.ipynb`.
4. Crear un repositorio GitHub.
5. Subir notebook, dataset y outputs.
6. Agregar un README con instrucciones de ejecución.